In [29]:
import os
import streamlit as st
from dotenv import load_dotenv
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

In [30]:
load_dotenv()
GOOGLE_API_KEY=os.getenv('GOOGLE_API_KEY')
if not GOOGLE_API_KEY:
    raise ValueError('GOOGLE API KEY IS NOT PRESENT IS MY .env FILE')

In [31]:
def read_pdfs(pdf_files):
    all_text= ""
    for pdf in pdf_files:
        reader= PdfReader(pdf)
        for page in reader.pages:
            text=page.extract_text()
            if text:
                all_text += text
    return all_text

In [32]:
text = read_pdfs(['attention.pdf'])

In [33]:
print(text)

Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring signiﬁcantly
less time to train. Our model a

In [34]:
def split_text(text):
    splitter=RecursiveCharacterTextSplitter(chunk_size= 600,chunk_overlap=200)
    return splitter.split_text(text)

In [35]:
split_text(text)

['Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best',
 'illia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispe

In [36]:
chunks=split_text(text)

In [37]:
len(chunks)

83

In [38]:
chunks[0]

'Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best'

In [39]:
chunks[1]

'illia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurrence and convolutions\nentirely. Experiments on two machine translation tasks show these models to\nbe superior in quality while being more parallelizable and requiring signiﬁcantly'

In [40]:
def create_embeddings():
    return HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

In [41]:
def build_vector_store(chunks):
    embedding= create_embeddings()
    documents= [Document(page_content=chunk) for chunk in chunks]
    vector_store= FAISS.from_documents(documents,embedding)
    vector_store.save_local('faiss_index')

In [42]:
build_vector_store(chunks)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9202.93it/s]


In [44]:
def load_vector_store():
    embeddings= create_embeddings()
    return FAISS.load_local('faiss_index',embeddings,allow_dangerous_deserialization=True)

In [46]:
def retrieve_chunks(question):
    vector_store= load_vector_store()
    return vector_store.similarity_search(question,k=10)

In [47]:
docs=retrieve_chunks('explain self attention in transformer')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7933.69it/s]


In [48]:
docs

[Document(id='f6de331e-da74-401a-88d1-bef710f4bed1', metadata={}, page_content='in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes\nit more difﬁcult to learn dependencies between distant positions [ 11]. In the Transformer this is\nreduced to a constant number of operations, albeit at the cost of reduced effective resolution due\nto averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as\ndescribed in section 3.2.\nSelf-attention, sometimes called intra-attention is an attention mechanism relating different positions'),
 Document(id='4e497c35-3d78-4dde-86a0-03b45f345d04', metadata={}, page_content='dk =dv =dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost\nis similar to that of single-head attention with full dimensionality.\n3.2.3 Applications of Attention in our Model\nThe Transformer uses multi-head attention in three different ways:\n• In "encoder-decoder atten

In [51]:
prompt = PromptTemplate(
    template="""
Answer the question using ONLY the context.

If the answer is not present, say:
"THE ANSWER IS NOT AVAILABLE IN THE PROVIDED CONTEXT."

Explain at Class 10 level.
Answer in bullet points.

Context:
{context}

Question:
{question}
""",
    input_variables=["context", "question"]
)

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0
)

chain = prompt | llm


def answer_question(question):

    docs = retrieve_chunks(question)

    context = "\n\n".join(doc.page_content for doc in docs)

    response = chain.invoke({
        "context": context,
        "question": question
    })

    return response.content

In [55]:
answer_question('what is Multi-Head Attention')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8456.75it/s]


ChatGoogleGenerativeAIError: Error calling model 'gemini-3.1-flash-lite' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource has been exhausted (e.g. check quota).', 'status': 'RESOURCE_EXHAUSTED'}}